# SKATER Geographic Clustering — Visual Results

Run this notebook after `python -m cluster_first.train` has completed.
It shows the Lyon gNodeB map for K = 5, 10, 20 and compares cluster sizes.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

REPO     = Path('../..')
GRAPHS   = REPO / 'data/graphs'
DATA     = REPO / 'data/cluster_first'
SWEEP_K  = [5, 10, 20]

bs_locs = pd.read_parquet(GRAPHS / 'bs_locations.parquet')
bs_locs['site_id'] = bs_locs['site_id'].astype(str)
print(f'Loaded {len(bs_locs)} BS locations')

## 1 — Lyon Map: K = 5 / 10 / 20 side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, K in zip(axes, SWEEP_K):
    asgn_path = DATA / f'K{K}' / 'cluster_assignments.parquet'
    if not asgn_path.exists():
        ax.set_title(f'K={K} — not trained yet')
        ax.axis('off')
        continue

    asgn = pd.read_parquet(asgn_path)
    asgn['site_id'] = asgn['site_id'].astype(str)
    df = asgn.merge(bs_locs, on='site_id', how='left')

    cmap = plt.cm.get_cmap('tab20', K)
    for k in range(K):
        sub = df[df['cluster_id'] == k]
        ax.scatter(sub['lon'], sub['lat'], s=10, color=cmap(k),
                   alpha=0.85, edgecolors='none', label=f'{k} (n={len(sub)})')

    ax.set_title(f'K = {K}  ({K} UPF zones)', fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(fontsize=6, ncol=2, loc='upper right', markerscale=2,
               title='cluster (n=BSs)', title_fontsize=7)

fig.suptitle('SKATER Geographic Clustering — Lyon gNodeBs', fontsize=14)
plt.tight_layout()
plt.savefig(REPO / 'figures/cluster_first/skater_map_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> figures/cluster_first/skater_map_comparison.png')

## 2 — Cluster Size Distribution per K

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, K in zip(axes, SWEEP_K):
    asgn_path = DATA / f'K{K}' / 'cluster_assignments.parquet'
    if not asgn_path.exists():
        ax.set_title(f'K={K} — not trained yet'); ax.axis('off'); continue

    asgn   = pd.read_parquet(asgn_path)
    sizes  = asgn.groupby('cluster_id').size().sort_values(ascending=False)
    colors = [plt.cm.tab20(i / K) for i in range(K)]

    ax.bar(range(len(sizes)), sizes.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(sizes.mean(), color='black', linewidth=1.2, linestyle='--',
                label=f'mean = {sizes.mean():.0f}')
    ax.set_title(f'K = {K}  |  min={sizes.min()}  max={sizes.max()}', fontsize=11)
    ax.set_xlabel('Cluster (sorted by size)')
    ax.set_ylabel('Number of BSs')
    ax.legend(fontsize=9)
    ax.set_xticks(range(len(sizes)))
    ax.set_xticklabels(sizes.index, fontsize=7, rotation=45)

fig.suptitle('Cluster Size Distribution — SKATER', fontsize=13)
plt.tight_layout()
plt.savefig(REPO / 'figures/cluster_first/skater_size_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 3 — Cluster Centroids on Map (K=10 focus)

In [ ]:
K_FOCUS = 10
asgn_path = DATA / f'K{K_FOCUS}' / 'cluster_assignments.parquet'

if not asgn_path.exists():
    print(f'K={K_FOCUS} not trained yet')
else:
    asgn = pd.read_parquet(asgn_path)
    asgn['site_id'] = asgn['site_id'].astype(str)
    df = asgn.merge(bs_locs, on='site_id', how='left')
    centroids = df.groupby('cluster_id')[['lat', 'lon']].mean()
    sizes     = df.groupby('cluster_id').size()

    cmap = plt.cm.get_cmap('tab20', K_FOCUS)
    fig, ax = plt.subplots(figsize=(10, 9))

    # All BSs (faint)
    for k in range(K_FOCUS):
        sub = df[df['cluster_id'] == k]
        ax.scatter(sub['lon'], sub['lat'], s=8, color=cmap(k), alpha=0.35, edgecolors='none')

    # Centroids (large markers with cluster label)
    for k, row in centroids.iterrows():
        ax.scatter(row['lon'], row['lat'], s=250, color=cmap(k),
                   edgecolors='black', linewidths=1.2, zorder=5)
        ax.annotate(f'C{k}\nn={sizes[k]}',
                    xy=(row['lon'], row['lat']),
                    xytext=(4, 4), textcoords='offset points',
                    fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec='none'))

    ax.set_title(f'K={K_FOCUS} SKATER Clusters — centroids labelled', fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(REPO / f'figures/cluster_first/skater_centroids_K{K_FOCUS}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## 4 — Cluster Traffic Signals (K=10)

In [ ]:
K_FOCUS = 10
series_path = DATA / f'K{K_FOCUS}' / 'cluster_series.npy'

if not series_path.exists():
    print(f'cluster_series not found for K={K_FOCUS} — run training first')
else:
    cs = np.load(series_path)          # (K, T_total)
    T  = cs.shape[1]
    t  = np.arange(T) / 96             # days from start

    cmap   = plt.cm.get_cmap('tab20', K_FOCUS)
    n_cols = 2
    n_rows = (K_FOCUS + 1) // 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows), squeeze=False)

    asgn  = pd.read_parquet(DATA / f'K{K_FOCUS}' / 'cluster_assignments.parquet')
    sizes = asgn.groupby('cluster_id').size()

    for k in range(K_FOCUS):
        ax = axes[k // n_cols][k % n_cols]
        ax.plot(t, cs[k], color=cmap(k), linewidth=0.7)
        ax.set_title(f'Cluster {k}  (n={sizes.get(k, "?")} BSs)', fontsize=9)
        ax.set_xlabel('Day')
        ax.set_ylabel('dl_norm sum')

    for k in range(K_FOCUS, n_rows * n_cols):
        axes[k // n_cols][k % n_cols].set_visible(False)

    fig.suptitle(f'77-day Aggregate Traffic per Cluster  (K={K_FOCUS})', fontsize=12)
    plt.tight_layout()
    plt.savefig(REPO / f'figures/cluster_first/skater_signals_K{K_FOCUS}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## 5 — Summary Table

In [ ]:
rows = []
for K in SWEEP_K:
    p = DATA / f'K{K}' / 'cluster_assignments.parquet'
    if not p.exists():
        continue
    sizes = pd.read_parquet(p).groupby('cluster_id').size()
    rows.append({
        'K':        K,
        'min_BSs':  int(sizes.min()),
        'max_BSs':  int(sizes.max()),
        'mean_BSs': round(sizes.mean(), 1),
        'std_BSs':  round(sizes.std(), 1),
        'singletons': int((sizes == 1).sum()),
    })

summary = pd.DataFrame(rows).set_index('K')
print(summary.to_string())
display(summary)